# Fase 2 — Cálculo de métricas

Lee los archivos `.pkl` de cada corrida y calcula **HV**, **GD** e **IGD**.

- **Frente de referencia:** unión global de todos los `fitnessF`, no-dominados.
- **Punto de referencia HV:** `[0, 0]` (consistente con el TFG).
- **Salida:** `results/metrics.csv`

In [ ]:
import os
import glob
import pickle
import re
import numpy as np
import pandas as pd

from pymoo.indicators.hv import Hypervolume
from pymoo.indicators.gd import GD
from pymoo.indicators.igd import IGD
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

RESULTS_BASE = './results'
N_VALUES     = [50, 100, 150, 200]
VARIANTES    = ['original', 'modified']
SEED_MODES   = ['same_seed', 'diff_seed']

## 1. Cargar todos los archivos `.pkl`

In [ ]:
def load_all_runs():
    runs = []
    for variante in VARIANTES:
        for seed_mode in SEED_MODES:
            for n in N_VALUES:
                folder = os.path.join(RESULTS_BASE, variante, seed_mode, f'N{n}')
                if not os.path.exists(folder):
                    continue
                pkls = sorted(glob.glob(os.path.join(folder, '*.pkl')))
                for i, pkl_path in enumerate(pkls):
                    with open(pkl_path, 'rb') as f:
                        data = pickle.load(f)
                    m = re.search(r'seed_(\d+)_', os.path.basename(pkl_path))
                    seed = int(m.group(1)) if m else None
                    runs.append({
                        'variante':  variante,
                        'seed_mode': seed_mode,
                        'N':         n,
                        'corrida':   i + 1,
                        'seed':      seed,
                        'fitnessF':  np.array(data['fitnessF']),
                    })
    return runs

runs = load_all_runs()
print(f'Total de corridas cargadas: {len(runs)}')

summary = pd.DataFrame([
    {'variante': r['variante'], 'seed_mode': r['seed_mode'], 'N': r['N']}
    for r in runs
]).groupby(['variante', 'seed_mode', 'N']).size().rename('corridas')
print(summary.to_string())

## 2. Construir frente de referencia global

In [ ]:
all_F = np.vstack([r['fitnessF'] for r in runs])
nds   = NonDominatedSorting()
front_idx = nds.do(all_F, only_non_dominated_front=True)
ref_front = all_F[front_idx]

print(f'Puntos totales (unión): {len(all_F)}')
print(f'Tamaño frente de referencia (no-dominados): {len(ref_front)}')

## 3. Calcular HV, GD e IGD por corrida

In [ ]:
hv_indicator  = Hypervolume(ref_point=np.zeros(2))
gd_indicator  = GD(ref_front)
igd_indicator = IGD(ref_front)

records = []
for r in runs:
    F = r['fitnessF']
    records.append({
        'variante':  r['variante'],
        'seed_mode': r['seed_mode'],
        'N':         r['N'],
        'corrida':   r['corrida'],
        'seed':      r['seed'],
        'HV':        hv_indicator.do(F),
        'GD':        gd_indicator.do(F),
        'IGD':       igd_indicator.do(F),
    })

metrics_df = pd.DataFrame(records)
out_path = os.path.join(RESULTS_BASE, 'metrics.csv')
metrics_df.to_csv(out_path, index=False)
print(f'Guardado: {os.path.abspath(out_path)}')
metrics_df.head()

## 4. Resumen estadístico por grupo

In [ ]:
metrics_df.groupby(['variante', 'seed_mode', 'N'])[['HV', 'GD', 'IGD']].agg(['mean', 'std', 'median'])